# Leaderboard Pipeline 00 — Project Contract and Submission Budget

This notebook freezes the working paths, verifies the completed 384 training output, records the environment, and protects the remaining submission budget.

You have used **1 of 3 submissions**. The remaining two submissions must pass OOF-based gates before being uploaded.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "leaderboard_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "scripts"))
DATA_ROOT = REPO_ROOT / "BDC2026"
MODEL_OUTPUT = REPO_ROOT / "outputs_dinov3_lora_384"
PIPELINE_ROOT = REPO_ROOT / "leaderboard_pipeline_outputs"
PIPELINE_ROOT.mkdir(parents=True, exist_ok=True)

LABEL2ID = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
DISPLAY_NAMES = {0: "Recyclable", 1: "Electronic", 2: "Organic"}

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("MODEL_OUTPUT:", MODEL_OUTPUT)
print("PIPELINE_ROOT:", PIPELINE_ROOT)


In [ ]:
import platform
import subprocess
import pandas as pd
import numpy as np
import torch
from importlib.metadata import version, PackageNotFoundError

STAGE_DIR = PIPELINE_ROOT / "00_config"
STAGE_DIR.mkdir(parents=True, exist_ok=True)

def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

def image_count(folder):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return len([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in exts]) if folder.exists() else 0

checks = {
    "train_total": sum(image_count(DATA_ROOT / "train" / name) for name in LABEL2ID),
    "test_total": image_count(DATA_ROOT / "test"),
    "submission_template": (DATA_ROOT / "submission.csv").exists(),
    "oof_predictions": (MODEL_OUTPUT / "oof_predictions.csv").exists(),
    "oof_probs": (MODEL_OUTPUT / "oof_probs.npy").exists(),
    "train_folds": (MODEL_OUTPUT / "train_folds.csv").exists(),
    "fold_histories": len(list(MODEL_OUTPUT.glob("fold*_history.csv"))),
    "fold_checkpoints": len(list(MODEL_OUTPUT.glob("fold*_best.pt"))),
}
checks


In [ ]:
assert checks["train_total"] == 26527, checks
assert checks["test_total"] == 1458, checks
assert checks["submission_template"], checks
assert checks["oof_predictions"], checks
assert checks["oof_probs"], checks
assert checks["fold_histories"] == 5, checks
assert checks["fold_checkpoints"] == 5, checks
print("Project contract: OK")


In [ ]:
try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
except Exception:
    git_commit = "unknown"

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "git_commit": git_commit,
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "cuda_devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else [],
    "packages": {name: package_version(name) for name in ["numpy", "pandas", "scikit-learn", "transformers", "peft", "torchvision"]},
}
(STAGE_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
(STAGE_DIR / "dataset_and_output_checks.json").write_text(json.dumps(checks, indent=2), encoding="utf-8")
environment


In [ ]:
budget_path = STAGE_DIR / "submission_budget.csv"
if budget_path.exists():
    submission_budget = pd.read_csv(budget_path)
else:
    submission_budget = pd.DataFrame([
        {"submission_id":"S01", "status":"submitted", "public_macro_f1_percent":98.42, "candidate":"384 baseline", "notes":"Rank 29 when recorded"},
        {"submission_id":"S02", "status":"reserved", "public_macro_f1_percent":np.nan, "candidate":"best cleaned duplicate-aware single model", "notes":"Use only after OOF gate"},
        {"submission_id":"S03", "status":"reserved", "public_macro_f1_percent":np.nan, "candidate":"OOF-optimized diverse ensemble", "notes":"Final submission"},
    ])
    submission_budget.to_csv(budget_path, index=False)
submission_budget


In [ ]:
submission_gate = {
    "S02": {
        "minimum_oof_macro_f1_gain": 0.0010,
        "minimum_folds_improved": 3,
        "weakest_class_f1_max_drop": 0.0005,
        "required": ["reproducible test probabilities", "submission schema validation"],
    },
    "S03": {
        "minimum_oof_gain_over_best_single": 0.0005,
        "minimum_folds_improved": 3,
        "required": ["model diversity analysis", "OOF-only ensemble weights", "submission schema validation"],
    },
    "policy": [
        "Never choose cleaning labels from leaderboard feedback.",
        "Never tune ensemble weights or class bias from leaderboard feedback.",
        "Reserve S03 for the strongest reproducible ensemble.",
    ],
}
(STAGE_DIR / "submission_gate.json").write_text(json.dumps(submission_gate, indent=2), encoding="utf-8")
submission_gate


## Decision

Do not upload another CSV yet. Complete notebooks 01–03, review the highest-priority labels, then train controlled candidates. Submission S02 should be the strongest single model that passes the gate; S03 should be the final OOF-optimized ensemble.
